# TuRBO-1入門：局所探索領域を更新するベイズ最適化

**TuRBO-1** は1つのTrust Region（局所探索領域）を持ち、実際の改善に応じて領域を広げたり狭めたりするベイズ最適化手法です。

このNotebookでは、初期観測 → `SingleTaskGP.fit()` → 局所範囲の計算 → 獲得関数による候補提案 → 実験 → `TuRBOState` の更新 → 次の反復を実行します。さらにThompson Samplingとの違い、探索領域と観測最良値の推移を可視化します。

**重要：** GP予測値だけでTuRBO状態を更新せず、実際に観測した値を使います。独自のTuRBO実装は作成せず、既存の `TuRBOState` と `TuRBOStrategy` を使用します。


## まず理解したい4つの役割

| 要素 | 役割 |
|---|---|
| GP | 観測データから目的関数の予測分布を学習 |
| 獲得関数 | 次に実験する候補の価値を計算 |
| Trust Region | 候補を探索する局所的な範囲を決める |
| TuRBO状態 | 改善履歴・領域長・成功/失敗カウンタを管理 |

TuRBOはGPの代替モデルではありません。**GPの予測を使いながら、候補を探す領域を適応的に変える仕組み**です。

## 1. Imports and reproducibility

In [ ]:
import torch
from botorch.acquisition.logei import qLogExpectedImprovement

from robotorchan.models import SingleTaskGP
from robotorchan.optim import TuRBOState, TuRBOStrategy

torch.set_default_dtype(torch.double)
torch.manual_seed(0)

### 例題の意味

ここでは温度・圧力に相当する二つの入力を、どちらも0〜1に正規化して扱います。目的関数は「目標条件からの距離の二乗にマイナスを付けた値」です。**0に近いほど良い品質**で、最大値は0です。実務では、この関数の代わりに実験装置やシミュレーターから観測値を取得します。

最初の10点は学習用の実測データに相当します。ここで求める`initial_center`は、最初の10点の中で最も良かった条件です。

In [ ]:
def objective(X: torch.Tensor) -> torch.Tensor:
    optimum = torch.tensor([0.72, 0.28])
    return -((X - optimum) ** 2).sum(dim=-1, keepdim=True)


dim = 2
bounds = torch.stack([torch.zeros(dim), torch.ones(dim)])
train_X = torch.rand(10, dim)
train_Y = objective(train_X)
best_index = train_Y.squeeze(-1).argmax()
initial_center = train_X[best_index].clone()
initial_best = float(train_Y[best_index].item())

print("initial center:", initial_center)
print("initial best:", initial_best)

### 初期データを可視化する

2次元の入力 `(x0,x1)` を探索します。真の最適条件は合成目的関数の定義に含まれますが、**GPはこの位置を知らず、観測点から推定します**。初期中心は観測済みで最も良い点です。

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(train_X[:, 0], train_X[:, 1], c=train_Y.squeeze(-1), cmap="viridis", s=65)
ax.scatter(initial_center[0], initial_center[1], marker="*", s=180, color="tab:red", label="Best initial observation")
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="x0", ylabel="x1", title="Initial observations and trust-region center")
ax.legend()
plt.show()

### 状態と探索範囲を分けて考える

`TuRBOState`は領域長、成功・失敗回数、最良値などを記憶します。`TuRBOStrategy`は中心座標と状態を使い、実際の探索範囲を作ります。**この段階では新しい実験を行いません。**

例えば全体範囲が0〜1、中心が0.5、領域長が0.8、方向の重みが1なら、局所範囲はおおむね0.1〜0.9です。中心が境界に近い場合は全体範囲で切り詰められます。下の出力の`local bounds`を確認してください。

In [ ]:
state = TuRBOState(
    dim=dim,
    batch_size=1,
    best_value=initial_best,
)
strategy = TuRBOStrategy(
    bounds,
    center=initial_center,
    state=state,
    num_restarts=3,
    raw_samples=64,
    seed=11,
)
local_bounds = strategy.trust_region_bounds()

assert local_bounds.shape == bounds.shape
assert torch.all(local_bounds >= bounds[0])
assert torch.all(local_bounds <= bounds[1])
print("trust-region length:", strategy.state.length)
print("local bounds:\n", local_bounds)

### 局所探索領域の形を確認する

`trust_region_bounds()` は各入力次元の下限・上限を返します。中心が境界に近い場合は、元の探索範囲 `[0,1]` で切り詰められます。

**Trust Regionは既知の候補制約を自動的に置き換えるものではありません。** 装置条件などの制約がある場合は別途指定します。

In [ ]:
from matplotlib.patches import Rectangle

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(train_X[:, 0], train_X[:, 1], s=35, label="Observed")
ax.scatter(initial_center[0], initial_center[1], marker="*", s=150, label="TR center")
ax.add_patch(Rectangle((local_bounds[0, 0].item(), local_bounds[0, 1].item()),
                       (local_bounds[1, 0] - local_bounds[0, 0]).item(),
                       (local_bounds[1, 1] - local_bounds[0, 1]).item(),
                       fill=False, edgecolor="tab:red", linewidth=2, label="Trust region"))
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="x0", ylabel="x1", title="Initial local search bounds")
ax.legend()
plt.show()

### GPと獲得関数の役割

`SingleTaskGP`は観測済みの`train_X`と`train_Y`から、未観測点の品質を予測します。`qLogExpectedImprovement`は「これまでの最高値を改善できそうか」を候補の価値として評価します。`strategy.optimize`は**全体範囲ではなく現在の局所範囲**で候補を選びます。

`candidate`は**予測に基づいて選ばれた未評価の入力**です。まだ品質の実測値ではありません。

### 予測・獲得関数・候補の区別

`SingleTaskGP.fit()` でGPを学習し、`qLogExpectedImprovement` で改善を評価します。`strategy.optimize()` は**現在の局所探索領域内**で候補を選びます。

この時点の `candidate` は**まだ実験していない入力**です。`result.metadata["trust_region_bounds"]` を確認して、局所範囲内にあることを検証します。

In [ ]:
model = SingleTaskGP(train_X, train_Y)
model.fit()
model.eval()
acquisition = qLogExpectedImprovement(
    model=model,
    best_f=train_Y.max(),
)
result = strategy.optimize(acquisition, q=1)
candidate = result.candidates.detach()
candidate_bounds = result.metadata["trust_region_bounds"]

assert candidate.shape == torch.Size([1, dim])
assert torch.all(candidate >= candidate_bounds[0])
assert torch.all(candidate <= candidate_bounds[1])
print("candidate:", candidate)
print("candidate-generation:", result.metadata["candidate_generation"])

### 候補の提案位置を図で確認する

提案された候補がTrust Region内にあることを可視化します。まだ実測していない段階では、候補を観測点と同一視しません。

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(train_X[:, 0], train_X[:, 1], s=30, label="Observed")
ax.scatter(candidate[:, 0], candidate[:, 1], marker="*", s=180, label="Proposed candidate")
ax.add_patch(Rectangle((candidate_bounds[0, 0].item(), candidate_bounds[0, 1].item()),
                       (candidate_bounds[1, 0] - candidate_bounds[0, 0]).item(),
                       (candidate_bounds[1, 1] - candidate_bounds[0, 1]).item(),
                       fill=False, edgecolor="tab:red", linewidth=2, label="Trust region"))
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel="x0", ylabel="x1", title="Candidate inside trust region")
ax.legend()
plt.show()

### ここで初めて実験を行う

`objective(candidate)`が実際の評価に相当します。その結果`new_Y`を受け取ってから`update_state`を呼びます。新しい結果が十分に良ければ成功カウンタが増え、そうでなければ失敗カウンタが増えます。成功・失敗が一定回数続くと領域長が変わります。

**観察ポイント**：`old_best`と新しい最良値、`old_center`と更新後の中心、二つのカウンタを比較してください。1回の実験で必ず領域長が変わるわけではありません。

In [ ]:
new_Y = objective(candidate)
old_center = strategy.center.clone()
old_best = strategy.state.best_value
strategy.update_state(new_Y, candidates=candidate)

train_X = torch.cat([train_X, candidate], dim=0)
train_Y = torch.cat([train_Y, new_Y], dim=0)

print("observed value:", new_Y.item())
print("best value:", old_best, "->", strategy.state.best_value)
print("center:", old_center, "->", strategy.center)
print("success counter:", strategy.state.success_counter)
print("failure counter:", strategy.state.failure_counter)

### 観測値でのみTuRBO状態を更新する

`strategy.update_state(new_Y, candidates=candidate)` は**実験で観測した `new_Y`** を受け取ります。GPの予測平均や獲得関数値を渡してはいけません。

その後 `train_X/train_Y` に観測を追加します。次の反復では追加済みデータでGPを再学習します。**TuRBO状態の更新とGP学習は別々の処理**です。

### 4反復の流れを追う

ここからは同じ処理を4回繰り返します。毎回、**最新の観測データでGPを学習し直す**ことが重要です。表示される各行は、順に「反復番号、領域長、状態の最良値、成功回数、失敗回数」です。

初期点やGPの予測によって、4回では領域の拡大・縮小が起きない場合もあります。これは異常ではありません。領域の変化には所定回数の連続した成功・失敗が必要です。

In [ ]:
history = []
for iteration in range(4):
    model = SingleTaskGP(train_X, train_Y)
    model.fit()
    model.eval()
    acquisition = qLogExpectedImprovement(
        model=model,
        best_f=train_Y.max(),
    )
    result = strategy.optimize(acquisition, q=1)
    new_X = result.candidates.detach()
    new_Y = objective(new_X)
    strategy.update_state(new_Y, candidates=new_X)
    train_X = torch.cat([train_X, new_X], dim=0)
    train_Y = torch.cat([train_Y, new_Y], dim=0)
    history.append(
        (
            iteration,
            strategy.state.length,
            strategy.state.best_value,
            strategy.state.success_counter,
            strategy.state.failure_counter,
        )
    )

for row in history:
    print(row)

### Trust Regionの幅と最良値の履歴を確認する

`history` には各反復の領域長、観測済み最良値、成功・失敗カウンタが入っています。**成功や失敗が所定回数連続しなければ、領域長が変化しないことも正常**です。

最良観測値は最大化問題なので悪化しませんが、各反復で改善するとは限りません。

In [ ]:
steps = [row[0] + 1 for row in history]
lengths = [row[1] for row in history]
best_values = [row[2] for row in history]
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(steps, lengths, "o-")
axes[0].set(xlabel="Additional iteration", ylabel="Trust-region length", title="Region size")
axes[1].plot(steps, best_values, "o-")
axes[1].axhline(initial_best, linestyle=":", color="gray", label="Initial best")
axes[1].set(xlabel="Additional iteration", ylabel="Best observed value", title="Incumbent")
axes[1].legend()
plt.show()

### 別の候補選択法：Thompson Sampling

獲得関数を最大化する代わりに、GPの事後分布から「あり得る品質関数」を抽選し、その関数上で良い候補を選ぶ方法です。**予測平均だけで決めない**ため、不確実な場所を選ぶこともあります。

下のセルは候補を生成し、品質を計算して表示する例です。この結果は学習データやTuRBO状態へ追加していません。次の反復へ反映する場合は、実測値を使った状態更新とデータ追加が必要です。

### Thompson Samplingによる候補選択

Thompson SamplingはGPの事後分布から関数をサンプリングし、そのサンプル上で良い候補を探す方法です。獲得関数を直接最適化する方法と異なります。

`strategy.thompson_sample()` は**同じTrust Region** を使います。生成した候補を観測しても、`update_state()` と学習データ追加を行わなければTuRBOの次の反復には反映されません。

In [ ]:
model = SingleTaskGP(train_X, train_Y)
model.fit()
model.eval()
ts_result = strategy.thompson_sample(
    model,
    q=1,
    n_candidates=128,
)
ts_candidate = ts_result.candidates
ts_bounds = ts_result.metadata["trust_region_bounds"]

assert ts_candidate.shape == torch.Size([1, dim])
assert torch.all(ts_candidate >= ts_bounds[0])
assert torch.all(ts_candidate <= ts_bounds[1])
assert ts_result.metadata["candidate_generation"] == "thompson"
print("Thompson candidate:", ts_candidate)

# Thompson候補も元の目的関数で評価します。
ts_observed_Y = objective(ts_candidate.detach())
assert ts_observed_Y.shape == torch.Size([1, 1])
print("Thompson candidate objective:", ts_observed_Y)


## 8. Trust-region state is not optimizer state

TuRBO stateが保持するのはincumbent、trust-region length、success/failure counters、restart情報です。AdamやL-BFGS等のnumerical optimizer stateとは別物です。

```text
TuRBO strategy = where to search
acquisition optimizer = how to optimize inside that region
surrogate model = what posterior to believe
acquisition = what candidate utility to optimize
```

### Restartは「やり直し」ではなく「探索場所の変更」

改善が得られず領域が小さくなりすぎた場合は、別の中心から再探索できます。Restartしても、過去に観測した最高値をなかったことにする必要はありません。一方、**GPの学習データをどう扱うかは別の設計判断**です。

## 10. High-dimensional and reduced models

TuRBOは既定でpublic/original input spaceを局所化します。PCA/PLS/RandomProjection modelが内部latent representationを使っても、candidateとtrust-region geometryを暗黙にlatentへ移しません。

public dimensionと1対1対応するARD lengthscaleがある場合はdimension weightsへ反映できます。一方、latent GP lengthscaleをraw-space geometryへ暗黙逆写像しません。

## 11. Async, robust, mixed, and multi-objective boundaries

- **Async**: unresolved evaluationsはacquisitionの`X_pending`。TuRBO stateはcompleted resultsだけで更新します。
- **Fantasization**: model/acquisition responsibility。fantasy batchはTuRBO `batch_size`ではありません。
- **Robust BO**: input perturbation scenario axisはtrust-region dimensionへ追加しません。必要なら`state_values`にrobust utilityを渡します。
- **Mixed**: continuous dimensionsだけをlengthで局所化し、integer/categorical合法値集合へ人工的な数値距離を導入しません。
- **Multi-objective**: qLogEHVI/qLogNEHVI等はlocal regionで最適化できますが、TuRBO-1 state updateにはcandidateごとのscalar state utilityが必要です。
- **Candidate constraints**: trust regionは既知input constraintsの代替ではありません。両方を同時に満たす必要があります。

### 学習の振り返り

1. GPの予測値と`objective`による実測値は、どこで区別されていましたか？
2. `length`を小さくすると、候補を探す範囲はどう変わりますか？
3. `q=1`のこの例で、成功・失敗は何点の実測結果を使って判定していますか？
4. Thompson Samplingで生成した候補は、自動的に学習データへ追加されますか？

**答え**：1. 実測は`objective`を呼んだ後です。2. 中心付近の狭い範囲になります。3. 1点です。4. いいえ、明示的な追加が必要です。

**次に読む**：[TuRBO-1とTuRBO-mの理論](../../docs/theory/optimization/10_trust_region.md) ／ [TuRBO-m Notebook](56_turbo_multi_benchmark.ipynb)。

## 12. TuRBO-1の実装上の確認事項

| 目的 | 既存API | 注意点 |
|---|---|---|
| GP学習 | `SingleTaskGP.fit()` | 観測を追加した後に再学習 |
| 状態の保持 | `TuRBOState` | 領域長・成功/失敗・最良値 |
| 局所範囲 | `TuRBOStrategy.trust_region_bounds()` | 元のbounds内に制限 |
| 候補提案 | `TuRBOStrategy.optimize()` | 既存獲得関数を局所的に最適化 |
| 状態更新 | `TuRBOStrategy.update_state()` | **実測値**を使う |
| 代替候補提案 | `TuRBOStrategy.thompson_sample()` | 観測結果の自動登録ではない |

**次に読む：** [TuRBO-1とTuRBO-mの理論](../../docs/theory/optimization/10_trust_region.md) ／ [TuRBO-m Notebook](56_turbo_multi_benchmark.ipynb)。